In [0]:
# Limpieza de widgets previos al inicio del proceso para evitar conflictos con ejecuciones anteriores
dbutils.widgets.removeAll()

In [0]:
# Importación de librerías de PySpark necesarias para transformaciones de columnas y definición de esquemas
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Definición de parámetros de entrada como widgets parametrizables
# Permite la ejecución dinámica desde Jobs sin modificar el código fuente
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "cat_ftr_smartdata_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsftrsmardatadev01")

In [0]:
# Recuperación de los valores de los widgets para usarlos como variables en el proceso
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

In [0]:
# Construcción de la ruta base en Azure Data Lake Storage Gen2 y ruta al archivo fuente CSV
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/"
ventas = ruta + "orders.csv"

In [0]:
# Definición explícita del esquema del archivo fuente para garantizar tipado correcto
# y evitar la inferencia automática, que puede generar inconsistencias en el tipo de datos
ventas_schema = StructType(fields=[StructField("ID_ORDER", StringType(), False),
                                  StructField("STORE", StringType(), False),
                                  StructField("REG", IntegerType(), True),
                                  StructField("DATE", DateType(), False),
                                  StructField("ID_ITEM", StringType(), False),
                                  StructField("QTY", IntegerType(), False),
                                  StructField("PRICE", DecimalType(12,4), False),
                                  StructField("DISCOUNT", DecimalType(12,4), False),
                                  StructField("ID_SELLER", StringType(), False),
                                  StructField("STATUS", StringType(), False)
                                  ])

In [0]:
%python
# Lectura del archivo CSV fuente desde ADLS
df_ventas = (spark.read.format("csv")
    .option("header", "true")
    .option("dateFormat", "dd/MM/yyyy")
    .schema(ventas_schema)
    .load(ventas))
#display(df_ventas)

In [0]:
# Enriquecimiento con campos de auditoría mínimos requeridos para capa bronze:
# - INGESTION_DATE    : fecha y hora exacta en que el registro fue cargado
# - SOURCE_FILE_NAME  : ruta del archivo fuente (trazabilidad de origen)
# - INGESTION_USER    : usuario o service principal que ejecutó la ingesta
df_ventas_ingestion_date = (df_ventas
    .withColumn("INGESTION_DATE", current_timestamp())
    .withColumn("SOURCE_FILE_NAME", col("_metadata.file_path"))
    .withColumn("INGESTION_USER", current_user())
)

In [0]:
# Selección y orden de columnas finales: campos de negocio seguidos de campos de auditoría
df_ventas_final = df_ventas_ingestion_date.select(
                                    "ID_ORDER",
                                    "STORE",
                                    "REG",
                                    "DATE",
                                    "ID_ITEM",
                                    "QTY",
                                    "PRICE",
                                    "DISCOUNT",
                                    "ID_SELLER",
                                    "STATUS",
                                    "INGESTION_DATE",
                                    "SOURCE_FILE_NAME",
                                    "INGESTION_USER"
                                )


In [0]:
# Escritura en la tabla Delta de la capa bronze con modo overwrite (carga completa)
# overwriteSchema=true permite actualizar el esquema si se agregan nuevas columnas
df_ventas_final.write.mode("overwrite").option("overwriteSchema", "true").insertInto(f"{catalogo}.{esquema}.ventas")

In [0]:
%sql    
-- Verificación de los datos cargados en la tabla bronze, incluyendo campos de auditoría
--select * from cat_ftr_smartdata_dev.bronze.ventas limit 10;
